In [1]:
import os
import asyncio
from typing import Any
from mcp.server.fastmcp import FastMCP
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

load_dotenv()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [13]:
class MultiServerMCPClient:
    def __init__(self, servers):
        self.servers = servers

    async def get_tools(self):
        """Mock tools from calculator server"""
        def mock_add(a: float, b: float) -> float:
            """Add 2 numbers"""
            return a+b
        
        def mock_multiply(a: float, b: float) -> float:
            """Multiply 2 numbers"""
            return a*b
        return [mock_add, mock_multiply]
    
llm = ChatOpenAI(
		model="gpt-4.1-mini",
		api_key=os.getenv("OPENAI_API_KEY"),
		base_url=os.getenv("OPENAI_API_BASE"),
		temperature=0
	)

client = MultiServerMCPClient(
    {
        "calculator": {
            "command": "python",
            "args": ["/root/code/lab-7.py"],
            "transport": "stdio",
        }
    }
)

async def run_agent_with_mcp():
    """Create and run agent with MCP tools"""
    tools = await client.get_tools()

    agent = create_agent(llm, tools)

    print("Agent created with MCP tools")
    print("Testing MCP-Integrated Agent:")
    print("=" * 60)

    print("\nTest 1:")
    math_response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What is 25 plus 17?"}]
    })

    print(f"Response: {math_response['messages'][-1].content}")

    print("\nTest 2:")
    multiply_response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "Calculate 8 times 9"}]
    })
    print(f"Response: {multiply_response['messages'][-1].content}")


    print("\nTest 3:")
    complex_response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What is (3+5) x 12"}]
    })
    print(f"Response: {complex_response['messages'][-1].content}")

    print("\nTest 4")
    general_response = await agent.ainvoke({
        "messages": [{"role":"user", "content": "What is the capital of France?"}]
    })

    print(f"Response: {general_response['messages'][-1].content}")


await run_agent_with_mcp()

Agent created with MCP tools
Testing MCP-Integrated Agent:

Test 1:
Response: 25 plus 17 is 42.

Test 2:
Response: 8 times 9 is 72.

Test 3:
Response: The result of (3+5) x 12 is 96.

Test 4
Response: The capital of France is Paris.
